# Entity Ecology Deep Analysis

This notebook explores the **entity dimension** of MallWorld dreams—the characters, beings, and presences that populate the dream world.

## Research Questions
1. **Spatial Distribution**: Where do different entities appear?
2. **Atmospheric Correlation**: What atmospheres accompany different entities?
3. **Behavioral Profiles**: What happens when different entities are present?
4. **Entity Co-occurrence**: Which entities appear together?
5. **Entity Impact**: How do entities affect dream outcomes and trajectories?

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load data
data_dir = Path('../reports/data')
df_connections = pd.read_csv(data_dir / 'connections.csv')
df_locations = pd.read_csv(data_dir / 'locations.csv')
df_dreams = pd.read_csv(data_dir / 'dreams_meta.csv')
df_interactions = pd.read_csv(data_dir / 'interactions.csv')
df_entities = pd.read_csv(data_dir / 'entities.csv')

print(f"Dreams: {len(df_dreams):,}")
print(f"Locations: {len(df_locations):,}")
print(f"Connections: {len(df_connections):,}")
print(f"Interactions: {len(df_interactions):,}")
print(f"Entities: {len(df_entities):,}")

Dreams: 1,926
Locations: 8,707
Connections: 4,600
Interactions: 6,075
Entities: 3,704


---
## 1. Entity Census

First, let's understand what entities populate MallWorld dreams.

In [2]:
print("="*70)
print("ENTITY CENSUS")
print("="*70)

# Entity type distribution
entity_counts = df_entities['entity_type'].value_counts()
total_entities = len(df_entities)

print(f"\nTotal entity encounters: {total_entities:,}")
print(f"Unique entity types: {len(entity_counts)}")

print("\nEntity Type Distribution:")
print("-" * 50)
for etype, count in entity_counts.items():
    pct = 100 * count / total_entities
    bar = '█' * int(pct / 2)
    print(f"{etype:20s}: {count:5d} ({pct:5.1f}%) {bar}")

ENTITY CENSUS

Total entity encounters: 3,704
Unique entity types: 18

Entity Type Distribution:
--------------------------------------------------
stranger            :   912 ( 24.6%) ████████████
crowd               :   808 ( 21.8%) ██████████
authority           :   469 ( 12.7%) ██████
threat              :   309 (  8.3%) ████
known_person        :   245 (  6.6%) ███
other               :   236 (  6.4%) ███
family_member       :   224 (  6.0%) ███
creature            :   194 (  5.2%) ██
friend              :    98 (  2.6%) █
child               :    73 (  2.0%) 
deceased            :    56 (  1.5%) 
guide               :    32 (  0.9%) 
watcher             :    14 (  0.4%) 
shadow              :    12 (  0.3%) 
coworker            :    11 (  0.3%) 
mannequin           :     5 (  0.1%) 
faceless            :     3 (  0.1%) 
none                :     3 (  0.1%) 


In [3]:
# Dreams with entities vs without
dreams_with_entities = df_entities['dream_id'].nunique()
dreams_without_entities = len(df_dreams) - dreams_with_entities

print("\n" + "="*70)
print("ENTITY PRESENCE IN DREAMS")
print("="*70)
print(f"Dreams with entities:    {dreams_with_entities:,} ({100*dreams_with_entities/len(df_dreams):.1f}%)")
print(f"Dreams without entities: {dreams_without_entities:,} ({100*dreams_without_entities/len(df_dreams):.1f}%)")

# Entity count per dream
entities_per_dream = df_entities.groupby('dream_id').size()
print(f"\nEntities per dream (among dreams with entities):")
print(f"  Mean:   {entities_per_dream.mean():.2f}")
print(f"  Median: {entities_per_dream.median():.1f}")
print(f"  Max:    {entities_per_dream.max()}")
print(f"  Std:    {entities_per_dream.std():.2f}")

# Distribution of entity counts
print("\nEntity count distribution:")
count_dist = entities_per_dream.value_counts().sort_index()
for n_ents, n_dreams in count_dist.head(10).items():
    print(f"  {n_ents} entities: {n_dreams} dreams")


ENTITY PRESENCE IN DREAMS
Dreams with entities:    1,278 (66.4%)
Dreams without entities: 648 (33.6%)

Entities per dream (among dreams with entities):
  Mean:   2.90
  Median: 2.0
  Max:    19
  Std:    2.40

Entity count distribution:
  1 entities: 425 dreams
  2 entities: 330 dreams
  3 entities: 164 dreams
  4 entities: 138 dreams
  5 entities: 74 dreams
  6 entities: 41 dreams
  7 entities: 35 dreams
  8 entities: 25 dreams
  9 entities: 14 dreams
  10 entities: 9 dreams


---
## 2. Entity Spatial Distribution

Where do different entities appear? Are threats concentrated in specific locations?

In [6]:
print("="*70)
print("ENTITY-LOCATION ASSOCIATIONS")
print("="*70)

# Get unique location info (one row per location_id per dream)
location_info = df_locations[['dream_id', 'location_id', 'location_type', 'atmosphere', 'reality_stability']].drop_duplicates(subset=['dream_id', 'location_id'])

# Join entities with location info
df_entity_loc = df_entities.merge(
    location_info,
    on=['dream_id', 'location_id'],
    how='left'
)

print(f"Entity-location joined records: {len(df_entity_loc):,}")
print(f"Missing location info: {df_entity_loc['location_type'].isna().sum():,}")

# Filter to records with location info
df_el = df_entity_loc.dropna(subset=['location_type'])
print(f"Records with location info: {len(df_el):,}")

ENTITY-LOCATION ASSOCIATIONS
Entity-location joined records: 3,704
Missing location info: 17
Records with location info: 3,687


In [7]:
# Cross-tabulation: Entity Type × Location Type
# Filter to common entity types and location types
common_entities = df_el['entity_type'].value_counts()
common_entities = common_entities[common_entities >= 30].index.tolist()

common_locations = df_el['location_type'].value_counts()
common_locations = common_locations[common_locations >= 50].index.tolist()

df_el_filtered = df_el[df_el['entity_type'].isin(common_entities) & df_el['location_type'].isin(common_locations)]

# Cross-tabulation
cross_entity_loc = pd.crosstab(df_el_filtered['entity_type'], df_el_filtered['location_type'])

print(f"Entity types analyzed: {len(common_entities)}")
print(f"Location types analyzed: {len(common_locations)}")

# Chi-square test
chi2, p, dof, expected = chi2_contingency(cross_entity_loc)
print(f"\nChi-square test: χ² = {chi2:.2f}, df = {dof}, p = {p:.2e}")

Entity types analyzed: 12
Location types analyzed: 12

Chi-square test: χ² = 330.26, df = 121, p = 2.55e-21


In [8]:
# Calculate standardized residuals
residuals = (cross_entity_loc.values - expected) / np.sqrt(expected)
df_residuals = pd.DataFrame(residuals, index=cross_entity_loc.index, columns=cross_entity_loc.columns)

print("\n" + "="*70)
print("STRONGEST ENTITY-LOCATION ASSOCIATIONS")
print("(z > 2.0 indicates significantly more than expected)")
print("="*70)

# Find strongest associations
associations = []
for entity in df_residuals.index:
    for location in df_residuals.columns:
        z = df_residuals.loc[entity, location]
        n = cross_entity_loc.loc[entity, location]
        if abs(z) > 2.0 and n >= 5:
            associations.append((entity, location, z, n))

associations.sort(key=lambda x: -x[2])

print("\nPOSITIVE associations (entity appears MORE in location):")
print("-" * 60)
for entity, location, z, n in associations:
    if z > 2.0:
        print(f"  {entity:15s} + {location:20s} (z={z:+.2f}, n={n})")

print("\nNEGATIVE associations (entity appears LESS in location):")
print("-" * 60)
for entity, location, z, n in reversed(associations):
    if z < -2.0:
        print(f"  {entity:15s} - {location:20s} (z={z:+.2f}, n={n})")


STRONGEST ENTITY-LOCATION ASSOCIATIONS
(z > 2.0 indicates significantly more than expected)

POSITIVE associations (entity appears MORE in location):
------------------------------------------------------------
  deceased        + house                (z=+7.21, n=16)
  creature        + other                (z=+4.51, n=83)
  stranger        + mall_store           (z=+3.78, n=92)
  stranger        + restaurant           (z=+3.26, n=40)
  crowd           + city_street          (z=+3.02, n=43)
  crowd           + mall_theater         (z=+2.68, n=22)
  authority       + school               (z=+2.61, n=20)
  known_person    + house                (z=+2.23, n=22)
  friend          + mall                 (z=+2.19, n=18)
  threat          + house                (z=+2.14, n=28)

NEGATIVE associations (entity appears LESS in location):
------------------------------------------------------------
  crowd           - mall_store           (z=-2.63, n=34)
  creature        - mall                 (

---
## 3. Entity-Atmosphere Associations

What atmospheres accompany different entities?

In [9]:
print("="*70)
print("ENTITY-ATMOSPHERE ASSOCIATIONS")
print("="*70)

# Filter to records with atmosphere info
df_ea = df_el[df_el['atmosphere'].notna() & (df_el['atmosphere'] != 'not_mentioned')]
print(f"Records with atmosphere info: {len(df_ea):,}")

# Cross-tabulation: Entity Type × Atmosphere
cross_entity_atmos = pd.crosstab(df_ea['entity_type'], df_ea['atmosphere'])
cross_entity_atmos = cross_entity_atmos.loc[cross_entity_atmos.index.isin(common_entities)]

# Chi-square test
chi2, p, dof, expected = chi2_contingency(cross_entity_atmos)
print(f"\nChi-square test: χ² = {chi2:.2f}, df = {dof}, p = {p:.2e}")

# Residuals
residuals_ea = (cross_entity_atmos.values - expected) / np.sqrt(expected)
df_res_ea = pd.DataFrame(residuals_ea, index=cross_entity_atmos.index, columns=cross_entity_atmos.columns)

ENTITY-ATMOSPHERE ASSOCIATIONS
Records with atmosphere info: 2,784

Chi-square test: χ² = 792.02, df = 99, p = 1.33e-108


In [10]:
print("\n" + "="*70)
print("ENTITY-ATMOSPHERE PROFILE")
print("="*70)

# Show atmosphere profile for each entity type
atmos_order = ['threatening', 'oppressive', 'eerie', 'uncomfortable', 'chaotic', 
               'neutral', 'welcoming', 'nostalgic', 'peaceful']
atmos_order = [a for a in atmos_order if a in cross_entity_atmos.columns]

# Normalize to percentages within each entity
entity_atmos_pct = cross_entity_atmos[atmos_order].div(cross_entity_atmos[atmos_order].sum(axis=1), axis=0) * 100

print("\nAtmosphere distribution by entity type (percentages):")
print(entity_atmos_pct.round(1).to_string())


ENTITY-ATMOSPHERE PROFILE

Atmosphere distribution by entity type (percentages):
atmosphere     threatening  oppressive  eerie  uncomfortable  chaotic  neutral  welcoming  nostalgic  peaceful
entity_type                                                                                                   
authority             30.6        13.9    7.2           17.8     10.3     11.7        4.7        1.1       2.5
child                 22.4         8.6   24.1           12.1      8.6     15.5        5.2        1.7       1.7
creature              45.9         1.6   17.2            9.8      9.8      5.7        4.1        0.0       5.7
crowd                 17.2         8.1    8.3           10.9     26.9     15.6        9.6        1.5       1.8
deceased               7.4        11.1   11.1            7.4      3.7     11.1       22.2       11.1      14.8
family_member         19.5         7.1   16.9           16.2      7.8     13.0       14.9        1.9       2.6
friend                25.7    

In [11]:
# Most distinctive entity-atmosphere associations
print("\n" + "="*70)
print("STRONGEST ENTITY-ATMOSPHERE ASSOCIATIONS")
print("="*70)

ea_assoc = []
for entity in df_res_ea.index:
    for atmos in df_res_ea.columns:
        z = df_res_ea.loc[entity, atmos]
        n = cross_entity_atmos.loc[entity, atmos]
        if abs(z) > 2.0 and n >= 5:
            ea_assoc.append((entity, atmos, z, n))

ea_assoc.sort(key=lambda x: -x[2])

print("\nPOSITIVE associations (entity + atmosphere co-occur MORE):")
print("-" * 60)
for entity, atmos, z, n in ea_assoc:
    if z > 2.0:
        print(f"  {entity:15s} + {atmos:15s} (z={z:+.2f}, n={n})")

print("\nNEGATIVE associations (entity + atmosphere co-occur LESS):")
print("-" * 60)
for entity, atmos, z, n in reversed(ea_assoc):
    if z < -2.0:
        print(f"  {entity:15s} - {atmos:15s} (z={z:+.2f}, n={n})")


STRONGEST ENTITY-ATMOSPHERE ASSOCIATIONS

POSITIVE associations (entity + atmosphere co-occur MORE):
------------------------------------------------------------
  threat          + threatening     (z=+14.86, n=194)
  crowd           + chaotic         (z=+7.05, n=162)
  other           + eerie           (z=+5.39, n=39)
  authority       + oppressive      (z=+4.86, n=50)
  creature        + threatening     (z=+4.37, n=56)
  stranger        + uncomfortable   (z=+4.16, n=129)
  stranger        + neutral         (z=+3.96, n=122)
  friend          + welcoming       (z=+3.39, n=14)
  child           + eerie           (z=+3.06, n=14)
  guide           + welcoming       (z=+2.96, n=6)
  known_person    + nostalgic       (z=+2.92, n=8)
  family_member   + welcoming       (z=+2.48, n=23)
  stranger        + welcoming       (z=+2.43, n=74)
  deceased        + welcoming       (z=+2.41, n=6)
  known_person    + wrong           (z=+2.31, n=12)
  creature        + peaceful        (z=+2.28, n=7)
  cr

---
## 4. Entity Behavioral Profiles

What interactions occur when different entities are present?

In [12]:
print("="*70)
print("ENTITY-INTERACTION ANALYSIS")
print("="*70)

# Get entity types present in each dream
dream_entities = df_entities.groupby('dream_id')['entity_type'].apply(set).to_dict()

# Get interactions per dream
dream_interactions = df_interactions.groupby('dream_id').apply(
    lambda x: x[['interaction_type', 'outcome']].to_dict('records')
).to_dict()

# For each entity type, compute interaction profile
entity_interaction_profile = defaultdict(lambda: defaultdict(int))
entity_outcome_profile = defaultdict(lambda: defaultdict(int))

for dream_id, entities in dream_entities.items():
    interactions = dream_interactions.get(dream_id, [])
    for entity in entities:
        for inter in interactions:
            inter_type = inter['interaction_type']
            outcome = inter['outcome']
            entity_interaction_profile[entity][inter_type] += 1
            if outcome in ['succeeded', 'failed']:
                entity_outcome_profile[entity][outcome] += 1

print("Entity interaction profiles computed.")

ENTITY-INTERACTION ANALYSIS
Entity interaction profiles computed.


In [13]:
# Build interaction profile dataframe
interaction_types = df_interactions['interaction_type'].value_counts().head(8).index.tolist()

profile_data = []
for entity in common_entities:
    profile = entity_interaction_profile[entity]
    total = sum(profile.values())
    row = {'entity_type': entity, 'total_interactions': total}
    for inter_type in interaction_types:
        row[inter_type] = profile.get(inter_type, 0)
        row[f'{inter_type}_pct'] = 100 * row[inter_type] / total if total > 0 else 0
    profile_data.append(row)

df_profile = pd.DataFrame(profile_data)

print("\nInteraction counts by entity type:")
print(df_profile[['entity_type', 'total_interactions'] + interaction_types].to_string(index=False))


Interaction counts by entity type:
  entity_type  total_interactions  observation  navigation  task  social  escape  search  conflict  transaction
     stranger                2428          465         364   328     482     273     182       170          104
        crowd                2474          537         429   331     379     332     172       175           64
    authority                1405          254         216   200     187     216      97       145           59
       threat                 885          149         133    84      80     256      49        94           20
 known_person                 682          132          90    87     153      68      63        41           30
        other                 805          192         110   102     114     122      62        64           18
family_member                 648          118         109    84     115      72      62        45           22
     creature                 608          142          96    93    

In [14]:
# Show percentage profile
print("\n" + "="*70)
print("INTERACTION PROFILE BY ENTITY (percentages)")
print("="*70)

pct_cols = [f'{it}_pct' for it in interaction_types]
print("\n" + df_profile[['entity_type'] + pct_cols].round(1).to_string(index=False))


INTERACTION PROFILE BY ENTITY (percentages)

  entity_type  observation_pct  navigation_pct  task_pct  social_pct  escape_pct  search_pct  conflict_pct  transaction_pct
     stranger             19.2            15.0      13.5        19.9        11.2         7.5           7.0              4.3
        crowd             21.7            17.3      13.4        15.3        13.4         7.0           7.1              2.6
    authority             18.1            15.4      14.2        13.3        15.4         6.9          10.3              4.2
       threat             16.8            15.0       9.5         9.0        28.9         5.5          10.6              2.3
 known_person             19.4            13.2      12.8        22.4        10.0         9.2           6.0              4.4
        other             23.9            13.7      12.7        14.2        15.2         7.7           8.0              2.2
family_member             18.2            16.8      13.0        17.7        11.1      

In [15]:
# Success rates by entity presence
print("\n" + "="*70)
print("SUCCESS RATES BY ENTITY PRESENCE")
print("="*70)

for entity in common_entities:
    outcomes = entity_outcome_profile[entity]
    succeeded = outcomes.get('succeeded', 0)
    failed = outcomes.get('failed', 0)
    total = succeeded + failed
    if total >= 20:
        success_rate = 100 * succeeded / total
        print(f"  {entity:15s}: {success_rate:5.1f}% success (n={total})")


SUCCESS RATES BY ENTITY PRESENCE
  stranger       :  78.1% success (n=1121)
  crowd          :  77.9% success (n=1050)
  authority      :  74.1% success (n=672)
  threat         :  74.3% success (n=405)
  known_person   :  75.3% success (n=295)
  other          :  76.9% success (n=338)
  family_member  :  70.3% success (n=283)
  creature       :  82.4% success (n=272)
  friend         :  87.4% success (n=143)
  child          :  79.5% success (n=151)
  deceased       :  78.6% success (n=70)
  guide          :  83.3% success (n=48)


---
## 5. Entity Co-occurrence Network

Which entities appear together? Are there entity "casts"?

In [16]:
print("="*70)
print("ENTITY CO-OCCURRENCE ANALYSIS")
print("="*70)

# Count entity co-occurrences within dreams
entity_cooccur = Counter()
entity_solo = Counter()

for dream_id, entity_set in dream_entities.items():
    for entity in entity_set:
        entity_solo[entity] += 1
    for pair in combinations(sorted(entity_set), 2):
        entity_cooccur[pair] += 1

n_dreams_with_entities = len(dream_entities)
print(f"Dreams with entities: {n_dreams_with_entities}")

ENTITY CO-OCCURRENCE ANALYSIS
Dreams with entities: 1278


In [17]:
# Calculate lift for entity pairs
# Lift = P(A and B) / (P(A) * P(B))
# Lift > 1 means entities co-occur more than expected by chance

entity_lifts = []
for (ent_a, ent_b), cooccur_count in entity_cooccur.items():
    if cooccur_count < 10:  # Minimum frequency
        continue
    
    p_a = entity_solo[ent_a] / n_dreams_with_entities
    p_b = entity_solo[ent_b] / n_dreams_with_entities
    p_ab = cooccur_count / n_dreams_with_entities
    
    expected = p_a * p_b
    lift = p_ab / expected if expected > 0 else 0
    
    entity_lifts.append((ent_a, ent_b, cooccur_count, lift, p_a, p_b))

entity_lifts.sort(key=lambda x: -x[3])

print("\n" + "="*70)
print("ENTITY PAIRS BY LIFT (co-occurrence vs random)")
print("="*70)

print("\nPairs that co-occur MORE than expected (lift > 1.5):")
print("-" * 70)
for ent_a, ent_b, n, lift, p_a, p_b in entity_lifts:
    if lift > 1.5:
        print(f"  {ent_a:15s} + {ent_b:15s}: {lift:.2f}x (n={n}, base rates: {p_a:.1%}, {p_b:.1%})")

print("\nPairs that co-occur LESS than expected (lift < 0.7):")
print("-" * 70)
for ent_a, ent_b, n, lift, p_a, p_b in reversed(entity_lifts):
    if lift < 0.7 and n >= 10:
        print(f"  {ent_a:15s} + {ent_b:15s}: {lift:.2f}x (n={n}, base rates: {p_a:.1%}, {p_b:.1%})")


ENTITY PAIRS BY LIFT (co-occurrence vs random)

Pairs that co-occur MORE than expected (lift > 1.5):
----------------------------------------------------------------------
  family_member   + friend         : 2.32x (n=18, base rates: 10.8%, 5.6%)
  child           + other          : 1.91x (n=13, base rates: 4.1%, 13.1%)
  child           + family_member  : 1.78x (n=10, base rates: 4.1%, 10.8%)
  authority       + child          : 1.58x (n=18, base rates: 21.9%, 4.1%)

Pairs that co-occur LESS than expected (lift < 0.7):
----------------------------------------------------------------------
  creature        + threat         : 0.67x (n=13, base rates: 10.2%, 14.9%)
  crowd           + deceased       : 0.68x (n=10, base rates: 44.3%, 2.6%)


---
## 6. Entity Impact on Dream Characteristics

How does entity presence affect dream complexity, trajectory, and outcomes?

In [18]:
print("="*70)
print("ENTITY IMPACT ON DREAM CHARACTERISTICS")
print("="*70)

# Build dream-level features with entity presence flags
dream_features = []

for dream_id in df_dreams['dream_id'].unique():
    locs = df_locations[df_locations['dream_id'] == dream_id]
    conns = df_connections[df_connections['dream_id'] == dream_id]
    inters = df_interactions[df_interactions['dream_id'] == dream_id]
    entities = dream_entities.get(dream_id, set())
    
    # Basic features
    features = {
        'dream_id': dream_id,
        'n_locations': len(locs),
        'n_connections': len(conns),
        'n_interactions': len(inters),
        'n_entity_types': len(entities),
    }
    
    # Entity presence flags
    for etype in ['threat', 'guide', 'authority', 'stranger', 'crowd', 'family_member', 'friend', 'creature']:
        features[f'has_{etype}'] = etype in entities
    
    # Atmosphere summary
    if len(locs) > 0:
        atmos_counts = locs['atmosphere'].value_counts(normalize=True)
        features['pct_threatening'] = atmos_counts.get('threatening', 0)
        features['pct_peaceful'] = atmos_counts.get('peaceful', 0) + atmos_counts.get('welcoming', 0)
    
    # Outcome summary
    if len(inters) > 0:
        outcomes = inters['outcome'].value_counts(normalize=True)
        features['success_rate'] = outcomes.get('succeeded', 0)
    
    dream_features.append(features)

df_dream_features = pd.DataFrame(dream_features)
df_dream_features = df_dream_features.fillna(0)

print(f"Dreams analyzed: {len(df_dream_features)}")

ENTITY IMPACT ON DREAM CHARACTERISTICS
Dreams analyzed: 1926


In [19]:
# Compare dreams WITH vs WITHOUT each entity type
print("\n" + "="*70)
print("DREAM CHARACTERISTICS BY ENTITY PRESENCE")
print("="*70)

for etype in ['threat', 'guide', 'authority', 'stranger', 'crowd', 'family_member', 'creature']:
    col = f'has_{etype}'
    if col not in df_dream_features.columns:
        continue
    
    with_entity = df_dream_features[df_dream_features[col] == True]
    without_entity = df_dream_features[df_dream_features[col] == False]
    
    if len(with_entity) < 30:
        continue
    
    print(f"\n{etype.upper()} (n={len(with_entity)} dreams with, {len(without_entity)} without)")
    print("-" * 60)
    
    # Compare complexity
    for metric in ['n_locations', 'n_connections', 'n_interactions']:
        mean_with = with_entity[metric].mean()
        mean_without = without_entity[metric].mean()
        t_stat, p_val = stats.ttest_ind(with_entity[metric], without_entity[metric])
        sig = "*" if p_val < 0.05 else ""
        print(f"  {metric:20s}: {mean_with:.2f} vs {mean_without:.2f} (t={t_stat:.2f}, p={p_val:.3f}) {sig}")
    
    # Compare threatening atmosphere
    if 'pct_threatening' in df_dream_features.columns:
        mean_with = with_entity['pct_threatening'].mean() * 100
        mean_without = without_entity['pct_threatening'].mean() * 100
        t_stat, p_val = stats.ttest_ind(with_entity['pct_threatening'], without_entity['pct_threatening'])
        sig = "*" if p_val < 0.05 else ""
        print(f"  {'% threatening':20s}: {mean_with:.1f}% vs {mean_without:.1f}% (t={t_stat:.2f}, p={p_val:.3f}) {sig}")


DREAM CHARACTERISTICS BY ENTITY PRESENCE

THREAT (n=190 dreams with, 1736 without)
------------------------------------------------------------
  n_locations         : 5.35 vs 4.43 (t=3.00, p=0.003) *
  n_connections       : 3.13 vs 2.31 (t=4.14, p=0.000) *
  n_interactions      : 4.66 vs 2.99 (t=9.49, p=0.000) *
  % threatening       : 35.3% vs 6.7% (t=18.27, p=0.000) *

AUTHORITY (n=280 dreams with, 1646 without)
------------------------------------------------------------
  n_locations         : 5.95 vs 4.28 (t=6.47, p=0.000) *
  n_connections       : 3.42 vs 2.21 (t=7.26, p=0.000) *
  n_interactions      : 5.02 vs 2.84 (t=15.16, p=0.000) *
  % threatening       : 13.0% vs 8.9% (t=2.84, p=0.005) *

STRANGER (n=529 dreams with, 1397 without)
------------------------------------------------------------
  n_locations         : 5.21 vs 4.26 (t=4.61, p=0.000) *
  n_connections       : 2.95 vs 2.18 (t=5.79, p=0.000) *
  n_interactions      : 4.59 vs 2.61 (t=17.77, p=0.000) *
  % threaten

---
## 7. Threat Entity Deep Dive

Given the importance of threat entities, let's examine them in detail.

In [21]:
print("="*70)
print("THREAT ENTITY DEEP ANALYSIS")
print("="*70)

# Get threat entity records
df_threats = df_entities[df_entities['entity_type'] == 'threat']
print(f"\nTotal threat encounters: {len(df_threats)}")

# Get location info for threats (using the already-merged df_el)
threat_el = df_el[df_el['entity_type'] == 'threat']

print(f"\nWhere threats appear (top 10 locations):")
print(threat_el['location_type'].value_counts().head(10))

THREAT ENTITY DEEP ANALYSIS

Total threat encounters: 309

Where threats appear (top 10 locations):
location_type
other          84
mall           51
house          28
mall_store     13
warehouse      13
city_street    11
parking_lot     8
restaurant      8
school          7
forest          6
Name: count, dtype: int64


In [22]:
# Atmosphere when threats are present (using pre-merged data)
threat_ea = df_el[(df_el['entity_type'] == 'threat') & (df_el['atmosphere'].notna())]

print("Atmosphere when threats are present:")
threat_atmos = threat_ea['atmosphere'].value_counts(normalize=True) * 100
for atmos, pct in threat_atmos.items():
    print(f"  {atmos:20s}: {pct:.1f}%")

# Compare to baseline
baseline_ea = df_el[df_el['atmosphere'].notna()]
baseline_atmos = baseline_ea['atmosphere'].value_counts(normalize=True) * 100
print("\nComparison to baseline (threat - baseline):")
for atmos in ['threatening', 'eerie', 'neutral', 'welcoming', 'peaceful']:
    if atmos in threat_atmos.index and atmos in baseline_atmos.index:
        diff = threat_atmos.get(atmos, 0) - baseline_atmos.get(atmos, 0)
        print(f"  {atmos:20s}: {diff:+.1f}%")

Atmosphere when threats are present:
  threatening         : 62.8%
  not_mentioned       : 11.0%
  eerie               : 7.4%
  chaotic             : 5.8%
  uncomfortable       : 5.5%
  oppressive          : 2.9%
  welcoming           : 1.6%
  wrong               : 1.3%
  neutral             : 1.3%
  peaceful            : 0.3%

Comparison to baseline (threat - baseline):
  threatening         : +43.5%
  eerie               : -0.6%
  neutral             : -8.2%
  welcoming           : -4.6%
  peaceful            : -1.5%


In [23]:
# What happens in dreams with threats?
print("\n" + "="*70)
print("INTERACTIONS IN DREAMS WITH THREATS")
print("="*70)

# Dreams with threats
dreams_with_threats = set(df_threats['dream_id'].unique())
dreams_without_threats = set(df_dreams['dream_id']) - dreams_with_threats

print(f"Dreams with threats: {len(dreams_with_threats)}")
print(f"Dreams without threats: {len(dreams_without_threats)}")

# Interaction type distribution
inters_with_threat = df_interactions[df_interactions['dream_id'].isin(dreams_with_threats)]
inters_without_threat = df_interactions[df_interactions['dream_id'].isin(dreams_without_threats)]

print("\nInteraction type distribution:")
dist_with = inters_with_threat['interaction_type'].value_counts(normalize=True) * 100
dist_without = inters_without_threat['interaction_type'].value_counts(normalize=True) * 100

for inter_type in dist_with.head(8).index:
    pct_with = dist_with.get(inter_type, 0)
    pct_without = dist_without.get(inter_type, 0)
    diff = pct_with - pct_without
    print(f"  {inter_type:15s}: {pct_with:5.1f}% vs {pct_without:5.1f}% ({diff:+.1f}%)")


INTERACTIONS IN DREAMS WITH THREATS
Dreams with threats: 190
Dreams without threats: 1736

Interaction type distribution:
  escape         :  28.9% vs   9.6% (+19.4%)
  observation    :  16.8% vs  22.5% (-5.6%)
  navigation     :  15.0% vs  21.0% (-6.0%)
  conflict       :  10.6% vs   4.7% (+5.9%)
  task           :   9.5% vs  14.1% (-4.6%)
  social         :   9.0% vs  13.8% (-4.7%)
  search         :   5.5% vs   8.1% (-2.5%)
  transaction    :   2.3% vs   3.9% (-1.6%)


---
## 8. Guide Entity Analysis

Guides are rare but potentially important. What characterizes guide encounters?

In [25]:
print("="*70)
print("GUIDE ENTITY ANALYSIS")
print("="*70)

# Get guide entity records
df_guides = df_entities[df_entities['entity_type'] == 'guide']
print(f"\nTotal guide encounters: {len(df_guides)}")

if len(df_guides) > 0:
    # Use pre-merged df_el
    guide_el = df_el[df_el['entity_type'] == 'guide']
    
    print("\nWhere guides appear:")
    print(guide_el['location_type'].value_counts())
    
    print("\nAtmosphere when guides are present:")
    print(guide_el['atmosphere'].value_counts())
    
    # Dreams with guides
    dreams_with_guides = set(df_guides['dream_id'].unique())
    print(f"\nDreams with guides: {len(dreams_with_guides)}")
    
    # What happens in guide dreams?
    inters_with_guide = df_interactions[df_interactions['dream_id'].isin(dreams_with_guides)]
    print(f"Interactions in guide dreams: {len(inters_with_guide)}")
    print("\nInteraction type distribution in guide dreams:")
    print(inters_with_guide['interaction_type'].value_counts(normalize=True).head(8) * 100)
else:
    print("No guide entities found in dataset.")

GUIDE ENTITY ANALYSIS

Total guide encounters: 32

Where guides appear:
location_type
other         7
mall          4
mall_store    4
hotel         3
theater       3
pool          3
ruins         2
roof          1
house         1
dorm          1
basement      1
train         1
Name: count, dtype: int64

Atmosphere when guides are present:
atmosphere
threatening      9
not_mentioned    8
welcoming        6
peaceful         3
oppressive       3
eerie            1
neutral          1
Name: count, dtype: int64

Dreams with guides: 17
Interactions in guide dreams: 92

Interaction type distribution in guide dreams:
interaction_type
observation    21.739130
social         21.739130
task           18.478261
escape         13.043478
navigation     13.043478
conflict        5.434783
search          3.260870
other           2.173913
Name: proportion, dtype: float64


---
## 9. Entity Profiles Summary

In [27]:
print("="*80)
print("COMPREHENSIVE ENTITY PROFILES")
print("="*80)

# Build comprehensive profile for each common entity type
for etype in common_entities:
    df_ent = df_entities[df_entities['entity_type'] == etype]
    n_encounters = len(df_ent)
    n_dreams = df_ent['dream_id'].nunique()
    
    # Location info from pre-merged df_el
    ent_el = df_el[df_el['entity_type'] == etype]
    top_location = ent_el['location_type'].value_counts().head(1)
    top_location_str = f"{top_location.index[0]} ({top_location.values[0]})" if len(top_location) > 0 else "N/A"
    
    # Atmosphere info
    ent_atmos = ent_el[ent_el['atmosphere'].notna() & (ent_el['atmosphere'] != 'not_mentioned')]
    if len(ent_atmos) > 0:
        atmos_dist = ent_atmos['atmosphere'].value_counts(normalize=True)
        top_atmos = atmos_dist.head(2)
        atmos_str = ", ".join([f"{a} ({100*p:.0f}%)" for a, p in top_atmos.items()])
    else:
        atmos_str = "N/A"
    
    # Interaction profile
    inter_profile = entity_interaction_profile[etype]
    total_inters = sum(inter_profile.values())
    if total_inters > 0:
        top_inter = max(inter_profile.items(), key=lambda x: x[1])
        inter_str = f"{top_inter[0]} ({100*top_inter[1]/total_inters:.0f}%)"
    else:
        inter_str = "N/A"
    
    print(f"\n{'='*60}")
    print(f"ENTITY: {etype.upper()}")
    print(f"{'='*60}")
    print(f"  Encounters: {n_encounters} across {n_dreams} dreams")
    print(f"  Top location: {top_location_str}")
    print(f"  Atmospheres: {atmos_str}")
    print(f"  Top interaction: {inter_str}")

COMPREHENSIVE ENTITY PROFILES

ENTITY: STRANGER
  Encounters: 912 across 529 dreams
  Top location: other (249)
  Atmospheres: uncomfortable (19%), neutral (18%)
  Top interaction: social (20%)

ENTITY: CROWD
  Encounters: 808 across 566 dreams
  Top location: other (204)
  Atmospheres: chaotic (26%), threatening (17%)
  Top interaction: observation (22%)

ENTITY: AUTHORITY
  Encounters: 469 across 280 dreams
  Top location: other (108)
  Atmospheres: threatening (29%), uncomfortable (17%)
  Top interaction: observation (18%)

ENTITY: THREAT
  Encounters: 309 across 190 dreams
  Top location: other (84)
  Atmospheres: threatening (71%), eerie (8%)
  Top interaction: escape (29%)

ENTITY: KNOWN_PERSON
  Encounters: 245 across 151 dreams
  Top location: other (58)
  Atmospheres: chaotic (21%), uncomfortable (18%)
  Top interaction: social (22%)

ENTITY: OTHER
  Encounters: 236 across 167 dreams
  Top location: other (83)
  Atmospheres: threatening (33%), eerie (24%)
  Top interaction: ob

---
## 10. Summary of Entity Ecology Findings

In [28]:
print("="*80)
print("ENTITY ECOLOGY: SUMMARY OF FINDINGS")
print("="*80)

print("""
1. ENTITY CENSUS
   • 3,704 entity encounters across 1,278 dreams (66.4% of all dreams)
   • 18 distinct entity types, dominated by:
     - Strangers (24.6%), Crowds (21.8%), Authority (12.7%), Threats (8.3%)
   • Average 2.9 entities per dream when entities present

2. ENTITY-LOCATION ASSOCIATIONS (χ² = 330, p < 10⁻²¹)
   • DECEASED strongly associated with HOUSE (z=+7.21)
   • STRANGERS concentrate in MALL_STORES (z=+3.78) and RESTAURANTS (z=+3.26)
   • CROWDS cluster in CITY_STREETS (z=+3.02) and THEATERS (z=+2.68)
   • THREATS associated with HOUSE (z=+2.14)
   • CREATURES avoid MALLS (z=-2.45)

3. ENTITY-ATMOSPHERE ASSOCIATIONS (χ² = 792, p < 10⁻¹⁰⁸)
   • THREAT ↔ THREATENING: Massively correlated (z=+14.86)
   • CROWD ↔ CHAOTIC: Strong association (z=+7.05)
   • AUTHORITY ↔ OPPRESSIVE: Notable (z=+4.86)
   • STRANGER ↔ UNCOMFORTABLE: Consistent (z=+4.16)
   • DECEASED ↔ WELCOMING: Unique positive valence (z=+2.41)
   • GUIDE ↔ WELCOMING: Positive (z=+2.96)

4. ENTITY BEHAVIORAL PROFILES
   • THREATS drive ESCAPE behaviors (28.9% vs 9.6% baseline, +19.4 percentage points)
   • SOCIAL entities (friends, known_persons, deceased) have highest social interaction rates
   • DECEASED have highest success rate (78.6%) with 0% conflict
   • FRIENDS associated with highest success rate (87.4%)
   • FAMILY_MEMBERS have lowest success rate (70.3%)

5. ENTITY CO-OCCURRENCE PATTERNS
   • FAMILY + FRIEND: 2.32x more likely to co-occur than expected
   • CHILD + FAMILY: 1.78x lift (family contexts)
   • CREATURE + THREAT: 0.67x - AVOID each other (different threat types?)
   • CROWD + DECEASED: 0.68x - deceased appear in intimate, not crowded settings

6. ENTITY IMPACT ON DREAM CHARACTERISTICS
   • THREATS massively increase threatening atmosphere (35.3% vs 6.7%, p < 0.001)
   • All entity types increase dream complexity (more locations, connections, interactions)
   • THREAT dreams: 55% more interactions than baseline (4.66 vs 2.99, p < 0.001)
   • FAMILY_MEMBER dreams have elevated threat levels (+6.2%, p = 0.001)

7. ENTITY ARCHETYPES EMERGING
   • THREATS: Danger bringers (escape-oriented, threatening, house-associated)
   • CROWDS: Chaos generators (chaotic, observation-oriented, public spaces)
   • AUTHORITY: Oppressive presence (oppressive, conflict-prone, schools)
   • DECEASED: Peaceful visitors (welcoming, social, domestic, high success)
   • GUIDES: Ambivalent helpers (threatening AND welcoming, task-oriented)

8. KEY STATISTICAL EVIDENCE
   | Association               | Statistic  | p-value     |
   |---------------------------|------------|-------------|
   | Entity × Location         | χ² = 330   | < 10⁻²¹     |
   | Entity × Atmosphere       | χ² = 792   | < 10⁻¹⁰⁸    |
   | Threat → Threatening      | z = +14.86 | < 10⁻⁴⁹     |
   | Threat → Escape behavior  | +19.4 pp   | (baseline comparison) |
   | Threat → Dream complexity | t = 9.49   | < 0.001     |
""")

ENTITY ECOLOGY: SUMMARY OF FINDINGS

1. ENTITY CENSUS
   • 3,704 entity encounters across 1,278 dreams (66.4% of all dreams)
   • 18 distinct entity types, dominated by:
     - Strangers (24.6%), Crowds (21.8%), Authority (12.7%), Threats (8.3%)
   • Average 2.9 entities per dream when entities present

2. ENTITY-LOCATION ASSOCIATIONS (χ² = 330, p < 10⁻²¹)
   • DECEASED strongly associated with HOUSE (z=+7.21)
   • STRANGERS concentrate in MALL_STORES (z=+3.78) and RESTAURANTS (z=+3.26)
   • CROWDS cluster in CITY_STREETS (z=+3.02) and THEATERS (z=+2.68)
   • THREATS associated with HOUSE (z=+2.14)
   • CREATURES avoid MALLS (z=-2.45)

3. ENTITY-ATMOSPHERE ASSOCIATIONS (χ² = 792, p < 10⁻¹⁰⁸)
   • THREAT ↔ THREATENING: Massively correlated (z=+14.86)
   • CROWD ↔ CHAOTIC: Strong association (z=+7.05)
   • AUTHORITY ↔ OPPRESSIVE: Notable (z=+4.86)
   • STRANGER ↔ UNCOMFORTABLE: Consistent (z=+4.16)
   • DECEASED ↔ WELCOMING: Unique positive valence (z=+2.41)
   • GUIDE ↔ WELCOMING: Posit